# Tutorial 03 (part 2): Autoencoders with fully connected layers
Prof Ivan Olier

## Introduction

In this tutorial, we implement autoencoders in TensorFlow and use them for two related tasks: reconstructing MNIST digits and visualising the learned latent space. We start with a simple shallow architecture, then move to a two-dimensional bottleneck, and finally test a deeper autoencoder.


We begin with the standard scientific Python libraries. `NumPy` and `Pandas` support data handling, while `Matplotlib` is used for the learning curves, image reconstructions, and latent-space plots.


In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

TensorFlow/Keras provides the building blocks for the autoencoders. We use the functional API through `Model`, define layers explicitly, and load MNIST as a compact image dataset for reconstruction.


In [ ]:
import tensorflow as tf
from tensorflow.keras.backend import clear_session
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model

Before training the models, it is useful to define a small plotting function for the training history. The training and validation losses will help us judge whether the model is learning smoothly or beginning to overfit.


In [ ]:
def plot_history(history):
    plt.figure()
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'])
    return;

MNIST contains 28 × 28 greyscale digit images. For a fully connected autoencoder, each image must be converted from a matrix into a vector. We also scale pixel intensities to the range `[0, 1]`, which matches the sigmoid output used later in the decoder.


In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.astype('float32') / 255.
X_test = X_test.astype('float32') / 255.
X_train = X_train.reshape((len(X_train), np.prod(X_train.shape[1:])))
X_test = X_test.reshape((len(X_test), np.prod(X_test.shape[1:])))
print(X_train.shape)
print(X_test.shape)

### The simplest version

We first build a minimal autoencoder with a single bottleneck layer. This gives us a baseline model that compresses each image and then reconstructs it from the compressed representation.


The input has 784 features, corresponding to the flattened 28 × 28 image. The encoder compresses this vector into 32 latent variables, and the decoder maps those variables back to 784 reconstructed pixel values.


In [ ]:
input_img = layers.Input(shape=(X_train.shape[1],))
bottleneck = layers.Dense(units=32, activation='relu', name='bottleneck')(input_img)
output_img = layers.Dense(units=X_train.shape[1], activation='sigmoid')(bottleneck)
model1 = Model(inputs=input_img, outputs=output_img)
model1.summary()

The reconstruction target is the input image itself. Binary cross-entropy is suitable here because the pixels have been scaled to `[0, 1]`, and Adam is a robust default optimiser for this type of neural network.


In [ ]:
model1.compile(loss='binary_crossentropy', optimizer='adam')

Training an autoencoder is different from ordinary supervised learning: the input and target are the same. The model is therefore learning to preserve the most important information needed to reconstruct each digit.


In [ ]:
history = model1.fit(X_train, X_train,
          batch_size=256, epochs=50,
          verbose=1,
          validation_split=0.2)

The loss curves show how reconstruction error changes during training. Ideally, both training and validation losses should decrease and remain close to each other.


In [ ]:
plot_history(history)

After training, we apply the autoencoder to unseen test images. The resulting predictions are reconstructed versions of the original digits.


In [ ]:
X_pred = model1.predict(X_test)

The next helper function displays original images and reconstructions side by side. The top row shows the true test images, and the bottom row shows the corresponding autoencoder outputs.


In [ ]:
def plot_digits(X_true, X_pred):
    n = X_pred.shape[0]  # how many digits we will display
    plt.figure(figsize=(20, 4))
    for i in range(n):
        # display original
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(X_true[i].reshape(28, 28))
        plt.gray()
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        # display reconstruction
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(X_pred[i].reshape(28, 28))
        plt.gray()
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
    plt.show()


Visual comparison is often the clearest way to assess reconstruction quality. Good reconstructions should preserve the main shape of each digit, even if some fine detail is lost.


In [ ]:
plot_digits(X_test, X_pred[0:10,:])

### Using the bottleneck for data visualisation

A bottleneck with only two latent variables is much more restrictive than the previous 32-dimensional representation. The benefit is that the learned codes can be plotted directly in two dimensions.


This second model compresses each digit into a two-dimensional latent vector. Such strong compression may reduce reconstruction quality, but it allows us to inspect whether similar digits occupy nearby regions of latent space.


In [ ]:
bottleneck = layers.Dense(units=2, name='bottleneck')(input_img)
output_img = layers.Dense(units=X_train.shape[1], activation='sigmoid')(bottleneck)
model2 = Model(inputs=input_img, outputs=output_img)
model2.summary()

The optimisation settings are kept the same as in the first model. This makes the change in behaviour easier to attribute to the smaller bottleneck rather than to a different loss or optimiser.


In [ ]:
model2.compile(loss='binary_crossentropy', optimizer='adam')

The model is trained again using the images as both inputs and targets. Because the bottleneck is now very small, the network must learn a compact two-variable summary of each digit.


In [ ]:
history = model2.fit(X_train, X_train,
          batch_size=256, epochs=50,
          verbose=1,
          validation_split=0.2)

The reconstructed test digits now reflect the cost of stronger compression. Comparing these examples with the previous model helps illustrate the trade-off between dimensionality reduction and reconstruction fidelity.


In [ ]:
X_pred = model2.predict(X_test)
plot_digits(X_test, X_pred[0:10,:])

To visualise the latent space, we separate the encoder from the full autoencoder. This encoder model maps each input image directly to its two-dimensional bottleneck representation.


In [ ]:
encoder_model = Model(inputs=input_img, outputs=bottleneck)
encoder_model.summary()

Each test image is projected into the learned two-dimensional space. Colouring the points by their true digit labels helps reveal whether the autoencoder has organised similar digits into related regions.


In [ ]:
X_test_encoded = encoder_model.predict(X_test)
plt.figure(figsize=(6, 6))
plt.scatter(X_test_encoded[:, 0], X_test_encoded[:, 1], c=y_test)
plt.colorbar()
plt.jet()
plt.show()

### Deep autoencoder

The previous model used a very simple encoder and decoder. We now add extra hidden layers, giving the network more capacity to learn non-linear transformations before and after the bottleneck.


The deep autoencoder first expands and compresses the representation through several dense layers, then reconstructs the image through a mirrored decoder structure. The bottleneck remains two-dimensional so that the latent space can still be plotted.


In [ ]:
x = layers.Dense(units=128, activation='relu')(input_img)
x = layers.Dense(units=32, activation='relu')(x)
bottleneck = layers.Dense(units=2, activation='relu', name='bottleneck')(x)
x = layers.Dense(units=32, activation='relu')(bottleneck)
x = layers.Dense(units=128, activation='relu')(x)
output_img = layers.Dense(units=X_train.shape[1], activation='sigmoid')(x)
model3 = Model(inputs=input_img, outputs=output_img)
model3.summary()

Although the architecture is deeper, the learning objective remains reconstruction. The loss still measures how close the output image is to the original input image.


In [ ]:
model3.compile(loss='binary_crossentropy', optimizer='adam')

This model has more parameters and may reconstruct the data better than the shallow two-dimensional autoencoder. The validation loss remains important, as a deeper network can also overfit more easily.


In [ ]:
history = model3.fit(X_train, X_train,
          batch_size=256, epochs=50,
          verbose=1,
          validation_split=0.2)

The test reconstructions allow us to judge whether the extra depth improved the visual quality of the decoded digits while still using only two latent variables.


In [ ]:
X_pred = model3.predict(X_test)
plot_digits(X_test[0:10,:], X_pred[0:10,:])

As before, the encoder component can be extracted from the trained autoencoder. This gives a direct mapping from each digit image to its two-dimensional latent representation.


In [ ]:
encoder_model = Model(inputs=input_img, outputs=bottleneck)
encoder_model.summary()

The final latent-space plot can be compared with the earlier two-dimensional model. Differences in clustering, overlap, and separation show how architecture affects the structure learned by the bottleneck.


In [ ]:
X_test_encoded = encoder_model.predict(X_test)
plt.figure(figsize=(6, 6))
plt.scatter(X_test_encoded[:, 0], X_test_encoded[:, 1], c=y_test)
plt.colorbar()
plt.jet()
plt.show()

## Exercises

1. We built three autoencoder models for the MNIST data. Which model best reconstructed the data? Which model gave the most useful visualisation of the data? Justify your answer using the reconstructions, learning curves, and latent-space plots.
2. Repeat the experiments using Fashion-MNIST. The dataset has the same image size as MNIST, so the same workflow can be reused.


Fashion-MNIST is a drop-in replacement for MNIST, but the images represent clothing items rather than handwritten digits. This makes it a useful test of whether the same autoencoder design generalises to a slightly more complex image dataset.


In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist

(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

Continue the Fashion-MNIST exercise below. A sensible starting point is to copy the MNIST preprocessing steps, train one or more of the autoencoders, and then compare reconstruction quality and latent-space organisation.
